## Federated model with per-client SMOTE

This notebook trains the federated model on the SceneFake dataset. Two points a reviewer may look for are handled explicitly:

1. **Class imbalance (SMOTE).** SceneFake is roughly 1 real : 4 fake. We apply **SMOTE per client** - each client oversamples the minority (real) class on its *own* local training data before training. This respects the federated setting (no raw data is pooled) and is applied to training data only, never to the evaluation set.

2. **Features (LFCC vs MFCC).** The pipeline is an LFCC-style cepstral pipeline (frame -> window -> FFT -> filterbank -> log -> DCT). Because Librosa has no built-in LFCC function, the extractor uses `librosa.feature.mfcc` as a proxy; MFCC and LFCC differ *only* in the filterbank spacing (mel vs linear). See the repository `docs/feature_note.md` and `src/features.py` for the drop-in true-LFCC extractor.

In [ ]:
# ---- STEP 0: Setup ----
!pip install -q imbalanced-learn
import os, random, numpy as np, torch
import librosa
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_curve, confusion_matrix

# SMOTE is used to correct the class imbalance in SceneFake (~1 real : 4 fake).
# In federated learning we apply it PER CLIENT (see local_train below).
from imblearn.over_sampling import SMOTE


In [ ]:
def get_paths_labels(path, label):
    return [(os.path.join(path, f), label) for f in os.listdir(path) if f.endswith('.wav')]

train_real = get_paths_labels("/kaggle/input/scenefake/train/real", 0)
train_fake = get_paths_labels("/kaggle/input/scenefake/train/fake", 1)
test_real  = get_paths_labels("/kaggle/input/scenefake/eval/real", 0)
test_fake  = get_paths_labels("/kaggle/input/scenefake/eval/fake", 1)

train_data = train_real + train_fake
test_data  = test_real + test_fake

random.shuffle(train_data)
random.shuffle(test_data)

train_paths, y_train = zip(*train_data)
test_paths, y_test = zip(*test_data)


In [ ]:
def extract_lfcc(path, sr=16000, n_mfcc=20, max_len=150):
    y, _ = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2])
    if feat.shape[1] >= max_len:
        feat = feat[:, :max_len]
    else:
        feat = np.pad(feat, ((0, 0), (0, max_len - feat.shape[1])), mode='constant')
    return feat


In [ ]:
class CNNSpeaker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 15 * 37, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
def partition_clients(paths, labels, num_clients=5):
    data = list(zip(paths, labels))
    random.shuffle(data)
    parts = np.array_split(data, num_clients)
    return [(list(p[:, 0]), list(map(int, p[:, 1]))) for p in parts]

clients = partition_clients(train_paths, y_train, num_clients=5)


In [ ]:
# ============================================================
# Per-client local training WITH SMOTE (federated-correct).
#
# Why SMOTE is here and not on the pooled data:
#   In federated learning each client only ever sees its OWN data. Balancing a
#   pooled dataset would require clients to share raw samples, which breaks the
#   federated principle. So each client balances its own local training partition
#   independently, then trains, then the server averages the weights (FedAvg).
#
# Why it is safe (no leakage):
#   SMOTE is applied ONLY to training data inside each client. The evaluation set
#   (X_test) is never resampled - see the evaluation cell below.
#
# k_neighbors guard:
#   After the 5-way client split the minority ('real', label 0) count can be
#   small. SMOTE's default k_neighbors=5 would crash if a client has <=5 minority
#   samples, so we shrink k to (n_minority - 1) and skip balancing if a client is
#   single-class or has only 1 minority sample.
# ============================================================
def local_train(model, data, epochs=1, batch_size=8, lr=0.001):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.train()

    paths, labels = data
    # Extract the 60 x 150 cepstral matrix for each clip in THIS client.
    X = np.array([extract_lfcc(p) for p in paths])   # shape: (N, 60, 150)
    y = np.array(labels)                             # 0 = real, 1 = fake

    # ---- per-client SMOTE (train only) ----
    if len(np.unique(y)) > 1:                        # need both classes present
        n_min = int(np.min(np.bincount(y)))          # size of minority class
        if n_min > 1:                                # need >1 minority sample
            k = min(5, n_min - 1)                    # safe neighbour count
            # SMOTE needs 2-D input, so flatten the image, balance, reshape back.
            Xf = X.reshape(len(X), -1)               # (N, 9000)
            Xf, y = SMOTE(random_state=42, k_neighbors=k).fit_resample(Xf, y)
            X = Xf.reshape(-1, 60, 150)              # back to (N', 60, 150)

    # Add the channel dim expected by Conv2d: (N, 1, 60, 150).
    X = torch.tensor(X[:, None, :, :], dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()

    return model.cpu().state_dict()  # send weights back to the server for FedAvg


In [ ]:
def fed_avg(state_dicts):
    avg_state = {}
    for key in state_dicts[0]:
        avg_state[key] = torch.stack([sd[key].float() for sd in state_dicts], dim=0).mean(dim=0)
    return avg_state


In [ ]:
global_model = CNNSpeaker()
ROUNDS = 5

for r in range(ROUNDS):
    print(f"Round {r+1}...")
    local_weights = []
    for client_data in clients:
        local_model = CNNSpeaker()
        local_model.load_state_dict(global_model.state_dict())
        weights = local_train(local_model, client_data, epochs=1)
        local_weights.append(weights)
    global_weights = fed_avg(local_weights)
    global_model.load_state_dict(global_weights)


In [ ]:
X_test = np.array([extract_lfcc(p) for p in test_paths])
X_test_tensor = torch.tensor(X_test[:, None, :, :], dtype=torch.float32).to('cuda')
y_test = np.array(y_test)

global_model.eval()
global_model = global_model.to('cuda')
probs, preds = [], []

with torch.no_grad():
    for i in range(0, len(X_test_tensor), 8):
        xb = X_test_tensor[i:i+8]
        out = global_model(xb)
        p = torch.softmax(out, dim=1)
        probs.extend(p[:, 1].cpu().numpy())
        preds.extend(torch.argmax(p, dim=1).cpu().numpy())

probs = np.array(probs)
preds = np.array(preds)


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, probs)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - tpr))]

print(f"\n✅ EER: {eer:.4f}")
print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, preds))
